# 06 — RQ2: Detect candidate defects in Agent Skill packages

**SkillGuard • MSR 2027 • reproducible screening, not confirmed defects**

This notebook applies `docs/rq2/rq2_defect_taxonomy_v0_1.json` to **exported skill folders from your existing Cisco sample**. It **does not run Cisco, run skill scripts, download repositories, or send network requests.**

It produces (1) evidence-backed **candidate warnings** for deterministic and screening rules, (2) a **coverage matrix** showing which taxonomy checks were *not* automated, (3) a researcher investigation worksheet, and (4) CSV/Markdown/LaTeX summaries.

**Research rules:** Do not use missing files from incomplete exports as evidence of defects; do not treat the absence of a warning as proof of quality; do not treat a regex match as a validated failure. Researcher investigation is required to validate candidates. This notebook is independent of Adam's Cisco security worksheet.

## 1 — Configuration

In [ ]:
SOURCE_RUN = "results/cisco_dataset_runs/agent_skills_sample/runs/2026-11-09_135034_sam-sample30-seed2027"
TAXONOMY_PATH = "docs/rq2/rq2_defect_taxonomy_v0_1.json"
OUTPUT_DIR = "results/rq2/seed2027_defect_screening"

MAX_EVIDENCE_CHARS = 200


## 2 — Find the repository and load taxonomy

Run from the SkillGuard repository (for example, from `notebooks/`). `PyYAML` improves YAML checks; if missing, YAML validity checks are **not evaluated** rather than being marked clean.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from urllib.parse import unquote
import ast, hashlib, json, os, re, sys, csv
import pandas as pd
from IPython.display import display

try:
    import yaml
except ImportError:
    yaml = None

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / "notebooks").is_dir() and (p / "results").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the SkillGuard repository (with notebooks/ and results/).")
RUN_DIR = (ROOT / SOURCE_RUN).resolve()
TAXONOMY_FILE = (ROOT / TAXONOMY_PATH).resolve()
OUT_DIR = (ROOT / OUTPUT_DIR).resolve()
if not RUN_DIR.is_dir():
    raise FileNotFoundError(f"Saved Cisco run directory not found: {RUN_DIR}")
if not TAXONOMY_FILE.is_file():
    raise FileNotFoundError(f"Taxonomy JSON not found: {TAXONOMY_FILE}. Put both v0.1 taxonomy files in docs/rq2/.")
TAXONOMY = json.loads(TAXONOMY_FILE.read_text(encoding="utf-8"))
ALL_RULES = TAXONOMY["spec_preflight"] + TAXONOMY["original_taxonomy"] + TAXONOMY["project_package_extensions"]
RULES_BY_ID = {r["id"]: r for r in ALL_RULES}
assert len(RULES_BY_ID) == 40, f"Expected 40 rules, got {len(RULES_BY_ID)}; taxonomy may have changed."
print("Repository:", ROOT)
print("Cisco export:", RUN_DIR)
print("Taxonomy version:", TAXONOMY["version"], "| rules:", len(ALL_RULES))
print("PyYAML:", "available" if yaml else "NOT installed; YAML validity check skipped")

## 3 — Inventory exported skills and verify completeness

Use the notebook's saved manifests as the source of truth. A missing exported file is a **data gap**, not necessarily a defect in the upstream skill. We only flag broken references when `folder_complete = True` and the exported files pass integrity checks.

In [ ]:
comparison_path = RUN_DIR / "context_comparison.csv"
comparisons = pd.read_csv(comparison_path, keep_default_na=False) if comparison_path.is_file() else pd.DataFrame()
metadata_by_skill = {}
if not comparisons.empty and "skill_id" in comparisons.columns:
    for skill_id, rows in comparisons.groupby("skill_id"):
        first = rows.iloc[0]
        metadata_by_skill[str(skill_id)] = {
            "repo": str(first.get("repo", "")),
            "skill_path": str(first.get("skill_path", "")),
            "stratum": str(first.get("sample_stratum", "")),
        }

SKILLS = []
for manifest_file in sorted(RUN_DIR.glob("*/manifest.json")):
    try:
        manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
    except (ValueError, OSError) as exc:
        print(f"Skipped invalid manifest {manifest_file}: {exc}")
        continue
    sid = manifest.get("skill_id", manifest_file.parent.name)
    folder = manifest_file.parent / "full"
    expected_files = manifest.get("files", [])
    errors = []
    for item in expected_files:
        rel = str(item.get("path", ""))
        dest = (folder / rel).resolve()
        if not dest.is_relative_to(folder.resolve()) or not dest.is_file():
            errors.append(f"missing_or_invalid_export:{rel}")
            continue
        sha = item.get("sha256", "")
        if sha and hashlib.sha256(dest.read_bytes()).hexdigest() != sha:
            errors.append(f"sha256_mismatch:{rel}")
    info = metadata_by_skill.get(sid, {})
    SKILLS.append({
        "skill_id": sid, "root": folder, "manifest": manifest,
        "folder_complete": bool(manifest.get("folder_complete", False)),
        "integrity_ok": not errors, "integrity_errors": "; ".join(errors),
        "exported_files": len(expected_files), **info,
    })
if not SKILLS:
    raise ValueError(f"No per-skill manifest.json found under {RUN_DIR}. Confirm the Cisco saved-run export.")
INV = pd.DataFrame([{k: v for k, v in s.items() if k not in ("root", "manifest")} for s in SKILLS])
display(INV[["skill_id", "folder_complete", "integrity_ok", "exported_files", "integrity_errors"]])
print(f"Found {len(SKILLS)} exported skills; complete: {INV.folder_complete.sum()}; integrity verified: {INV.integrity_ok.sum()}")

## 4 — Detector primitives and evidence safeguards

Only text inspection, regular expressions, and Python AST parsing occur. **No imported skill module or script is executed.** Suspected credentials and literal home directory segments are masked in published excerpts.

In [ ]:
EMAILISH_SECRET = re.compile(r"(?i)(password|secret|token|api[_-]?key)\s*[:=]\s*['\"][^'\"]{4,}['\"]")
API_TOKEN = re.compile(r"\b(?:AKIA|ASIA|ghp_|gho_|github_pat_|sk-)[A-Za-z0-9_-]{8,}")
HOME_PATH = re.compile(r"(?<![\w/])(/Users|/home)/[A-Za-z0-9_.-]+")
URL_RE = re.compile(r"^(?:https?://|mailto:|data:|#|//)", re.I)
CODE_LINK_RE = re.compile(r"(?<![\w./-])(?:\.?/?(?:scripts|references|assets|templates|resources)/)[A-Za-z0-9_./-]+")
MARKDOWN_LINK_RE = re.compile(r"(?<!!)\[[^\]]+\]\(([^)\s]+)(?:\s+['\"][^)]*['\"])?\)")

CANDIDATES = []
RUN_CHECKS = set()

def clean_excerpt(text):
    text = EMAILISH_SECRET.sub("[REDACTED_CREDENTIAL]", str(text))
    text = API_TOKEN.sub("[REDACTED_TOKEN]", text)
    text = HOME_PATH.sub(r"\1/[REDACTED_USER]", text)
    return text.strip()[:MAX_EVIDENCE_CHARS]

def read_text_file(skill, rel):
    root = skill["root"].resolve()
    path = (root / rel).resolve()
    if not path.is_relative_to(root) or not path.is_file() or path.is_symlink():
        return None
    if path.stat().st_size > 2_000_000:  # conservative cap
        return None
    return path.read_text(encoding="utf-8", errors="replace")

def emit(skill, rule_id, rel, line, signal, excerpt="", rationale="", check="static_screening"):
    if rule_id not in RULES_BY_ID:
        raise ValueError(f"Unknown taxonomy rule: {rule_id}")
    CANDIDATES.append({
        "skill_id": skill["skill_id"], "repo": skill.get("repo", ""),
        "skill_path": skill.get("skill_path", ""), "stratum": skill.get("stratum", ""),
        "rule_id": rule_id, "rule_name": RULES_BY_ID[rule_id]["name"],
        "rule_type": RULES_BY_ID[rule_id].get("rule_type", "spec" if rule_id.startswith("S") else "package"),
        "file_path": str(rel), "line_number": int(line or 0),
        "signal": clean_excerpt(signal), "supporting_excerpt": clean_excerpt(excerpt),
        "analysis_note": rationale, "detection_method": check,
        "verification_level": "source_verified" if excerpt else "textual_only",
        "candidate_status": "candidate", "impact": "", "recommended_fix": "", "investigation_notes": "",
    })

def line_at(text, pos):
    return text.count("\n", 0, pos) + 1

def body_without_code_fences(body):
    lines = body.splitlines()
    in_fence = False
    output = []
    for l in lines:
        if re.match(r"^\s*(`{3,}|~{3,})", l):
            in_fence = not in_fence
            continue
        if not in_fence:
            output.append(l)
    return "\n".join(output)

def frontmatter(text):
    lines = text.lstrip("\ufeff").splitlines()
    if not lines or lines[0].strip() != "---":
        return None, text, "missing frontmatter opening delimiter"
    closing = next((i for i in range(1, len(lines)) if lines[i].strip() == "---"), None)
    if closing is None:
        return None, text, "missing frontmatter closing delimiter"
    yaml_text = "\n".join(lines[1:closing]); body = "\n".join(lines[closing+1:])
    if yaml is None:
        simple = {}
        for line in lines[1:closing]:
            m = re.match(r"^([\w-]+)\s*:\s*(.*)$", line)
            if m:
                simple[m.group(1)] = m.group(2).strip().strip("'\"")
        return simple, body, "YAML parser unavailable: simple scalar extraction only"
    try:
        obj = yaml.safe_load(yaml_text)
        if not isinstance(obj, dict):
            return None, body, "frontmatter is not a YAML mapping"
        return obj, body, ""
    except yaml.YAMLError:
        return None, body, "invalid YAML mapping"

print("Static analysis helpers loaded (no skill code execution).")

## 5 — Preflight and SKILL.md-only checks

Rule thresholds come from `docs/rq2/` and are **project operationalizations**, not all mandatory specification constraints. `R2.4` and similar style rules should be validated as quality heuristics, not called exploits.

In [ ]:
for skill in SKILLS:
    skill_id = skill["skill_id"]
    source = read_text_file(skill, "SKILL.md")
    if source is None:
        if skill["folder_complete"] and skill["integrity_ok"]:
            emit(skill, "S1", "SKILL.md", 0, "SKILL.md absent from complete, verified export", rationale="Confirm the source folder and snapshot before calling it a spec violation")
        continue
    metadata, body, error = frontmatter(source)
    if metadata is None:
        emit(skill, "S2", "SKILL.md", 1, "Invalid/missing YAML frontmatter", error)
        continue
    if yaml is not None:
        RUN_CHECKS.add("S2")
    d = str(metadata.get("description") or "").strip()
    name = str(metadata.get("name") or "").strip()
    if not d:
        emit(skill, "R1.1", "SKILL.md", 1, "Missing description", "description is blank or absent")
    if d and len(d) < 30:
        emit(skill, "R1.2", "SKILL.md", 1, f"Description only {len(d)} characters", d)
    if len(d) > 300:
        emit(skill, "R1.5", "SKILL.md", 1, f"Description has {len(d)} characters (>300, style heuristic)", d)
    if not d or len(d) > 1024:
        emit(skill, "S4", "SKILL.md", 1, "Missing/overlong description", f"length={len(d)}")
    # Compare with original skill directory, not the exported skill_id folder which has a hash suffix.
    original = str(skill.get("skill_path", ""))
    actual_dir = Path(original).parent.name if original.endswith("SKILL.md") else ""
    valid_name = bool(re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)*", name)) and len(name) <= 64
    if not valid_name or (actual_dir and name != actual_dir):
        emit(skill, "S3", "SKILL.md", 1, "Malformed or path-mismatched skill name", f"Name={name}; original directory={actual_dir or '(not in comparison)'}")
    b_lines = body.splitlines()
    if len(b_lines) > 500:
        emit(skill, "R2.1", "SKILL.md", 1, f"{len(b_lines)} Markdown body lines (>500)")
    headers = [(i+1, l) for i,l in enumerate(b_lines) if re.match(r"^#\s+", l)]
    if headers and name:
        h = headers[0][1].removeprefix("#").strip().lower()
        normalize = lambda t: re.sub(r"[\s_-]+", " ", str(t).lower()).strip()
        if normalize(h) == normalize(name):
            emit(skill, "R2.4", "SKILL.md", headers[0][0], "First H1 duplicates skill name", headers[0][1])
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", body) if p.strip() and not p.lstrip().startswith("#")]
    if d and paragraphs:
        norm = lambda x: re.sub(r"\s+", " ", x).strip().casefold()
        if norm(d) == norm(paragraphs[0]):
            emit(skill, "R2.5", "SKILL.md", 1, "Body first paragraph duplicates description", paragraphs[0])
    nonblank, fenced_lines, fences = 0, 0, 0
    in_fence = False
    for l in b_lines:
        if re.match(r"^\s*(`{3,}|~{3,})", l):
            fences += 1
            in_fence = not in_fence
            continue
        if l.strip():
            nonblank += 1
            if in_fence: fenced_lines += 1
    ratio = (fenced_lines / nonblank) if nonblank else 0
    if nonblank and ratio > .6:
        emit(skill, "R3.1", "SKILL.md", 1, f"Fenced code represents {ratio:.0%} of nonblank body lines")
    if fences // 2 > 8:
        emit(skill, "R3.2", "SKILL.md", 1, f"{fences//2} fenced code blocks (>8)")
    if len(b_lines) > 300:
        resources = [i for i in skill["manifest"].get("files",[]) if str(i.get("path", "")).startswith(("scripts/", "references/"))]
        if not resources and skill["folder_complete"]:
            emit(skill, "R3.3", "SKILL.md", 1, "Large skill without scripts/references", rationale="Screening heuristic; standalone content may be justified")
    body_prose = body_without_code_fences(body)
    for m in list(re.finditer(r"(?im)^\s*#{1,6}\s*(?:changelog|change\s+log|version\s+history)\b", body_prose))[:1]:
        emit(skill, "R4.2", "SKILL.md", line_at(body_prose,m.start()), "Changelog heading in active instructions", m.group(0))
    for m in list(re.finditer(r"\b(?:TODO|FIXME|TBD)\b", body_prose))[:3]:
        emit(skill, "R4.4", "SKILL.md", line_at(body_prose,m.start()), "Unfinished marker in prose", m.group(0), "Review whether it is an example, quotation or unfinished task")
    for m in list(re.finditer(r"(?i)(?:verify\s*=\s*False|--no-verify|curl\s+-k\b|rejectUnauthorized\s*:\s*false)", source))[:3]:
        emit(skill,"R5.2","SKILL.md",line_at(source,m.start()),"Safety verification bypass pattern",m.group(0),"Determine whether this is active instruction or merely an example")
    for m in list(re.finditer(r"(?<!\w)/(?:Users|home)/[A-Za-z0-9_.-]+/", source))[:2]:
        emit(skill,"R5.6","SKILL.md",line_at(source,m.start()),"Literal user-specific absolute path","[USER_HOME_PATH_REDACTED]", "Verify if the path is a realistic example, not an actual leaked path")
    RUN_CHECKS.update(["S1", "S3", "S4", "R1.1", "R1.2", "R1.5", "R2.1", "R2.4", "R2.5", "R3.1", "R3.2", "R3.3", "R4.2", "R4.4", "R5.2", "R5.6"])
print("Completed SKILL.md checks:", len(CANDIDATES), "candidates")

## 6 — Full-folder checks: broken references, dependencies, side effects

**Safety and scope:** Broken-reference warnings are only generated when the manifest says the export is complete **and** its hashes match. Dependency and side-effect patterns are *triage signals*, not proven problems. Scripts are **read and parsed only**, never imported or executed.

In [ ]:
def reference_targets(text):
    targets=[]
    for m in MARKDOWN_LINK_RE.finditer(text):
        targets.append((m.group(1),line_at(text,m.start()),"Markdown link"))
    for m in CODE_LINK_RE.finditer(text):
        targets.append((m.group(0),line_at(text,m.start()),"explicit resource path"))
    return targets

def looks_like_resource(ref):
    v = unquote(ref).split("#",1)[0].split("?",1)[0].strip().strip("`'\".,;:")
    if not v or URL_RE.search(v) or any(x in v for x in ("${", "{", "}", "<", ">", "*", "...")):
        return ""
    if v.startswith("/") or v.startswith("~") or "\\" in v:
        return ""
    return v

STDLIB = getattr(sys, "stdlib_module_names", set())
DEPENDENCY_ALIASES = {"PIL":"pillow", "cv2":"opencv-python", "yaml":"pyyaml", "bs4":"beautifulsoup4"}
SIDE_EFFECT_REGEX = re.compile(r"(?:\bcrontab\b|\bos\.kill\s*\(|\bshutil\.rmtree\s*\(|\bos\.remove\s*\(|\bPath\([^)]*\)\.unlink\s*\()")
OS_COMMAND_REGEX = re.compile(r"(?i)(?:\b(?:pbcopy|apt-get|crontab|launchctl|osascript|powershell\.exe|brew)\b)")

for skill in SKILLS:
    if not skill["integrity_ok"]:
        continue  # Don't analyze modified/missing exported source as if verified.
    manifest_files = [str(f.get("path", "")) for f in skill["manifest"].get("files", [])]
    root = skill["root"].resolve()
    if skill["folder_complete"]:
        for rel in manifest_files:
            if not (rel == "SKILL.md" or rel.lower().endswith(".md")):
                continue
            text = read_text_file(skill, rel)
            if text is None: continue
            observed=set()
            for raw, lineno, why in reference_targets(text):
                target = looks_like_resource(raw)
                if not target or (target,lineno) in observed: continue
                observed.add((target,lineno))
                # Markdown links resolve relative to current file; docs sometimes write skill-root paths.
                opts = [(root / Path(rel).parent / target).resolve(), (root / target).resolve()]
                if not any(p.is_relative_to(root) and p.exists() for p in opts):
                    # Avoid flagging external relative links or generated example outputs as defects.
                    if not (target.startswith(("./","../","scripts/","references/","assets/","templates/","resources/")) or
                            ("/" not in target and Path(target).suffix.lower() in (".md", ".py", ".sh", ".json"))):
                        continue
                    emit(skill, "P1", rel, lineno, f"Unresolved {why}: {target}", raw,
                         "Manifest is complete and integrity-verified. Manually exclude generated or intentionally external files.")
        RUN_CHECKS.add("P1")
    # Script-only checks, always screening rather than confirmation.
    skill_markdown = read_text_file(skill,"SKILL.md") or ""
    manifest_text = "\n".join(read_text_file(skill, f) or "" for f in manifest_files if Path(f).name in
                                  ("requirements.txt", "pyproject.toml", "Pipfile", "environment.yml", "package.json"))
    package_declarations = (skill_markdown + "\n" + manifest_text).lower()
    for rel in manifest_files:
        if not rel.endswith(".py"): continue
        text = read_text_file(skill,rel)
        if text is None: continue
        # Literal imports only: AST parsing is safe and does not execute code.
        try: tree=ast.parse(text)
        except SyntaxError: tree=None
        third_party=set()
        if tree:
            for n in ast.walk(tree):
                if isinstance(n, ast.Import):
                    third_party.update(alias.name.split(".",1)[0] for alias in n.names)
                elif isinstance(n, ast.ImportFrom) and n.module and n.level==0:
                    third_party.add(n.module.split(".",1)[0])
        third_party={x for x in third_party if x not in STDLIB and x not in {"__future__"} and not x.startswith("_")}
        undocumented=[x for x in sorted(third_party) if x.lower() not in package_declarations and DEPENDENCY_ALIASES.get(x,"").lower() not in package_declarations]
        if undocumented:
            emit(skill, "P2", rel, 1, f"Possible undocumented imports: {', '.join(undocumented[:8])}",
                 rationale="Review imports and repo dependencies; local modules and optional packages may be intentionally bundled.", check="heuristic_screening")
        for m in list(SIDE_EFFECT_REGEX.finditer(text))[:4]:
            line = line_at(text,m.start())
            excerpt = text.splitlines()[line-1] if line <= len(text.splitlines()) else m.group(0)
            emit(skill, "P4", rel, line, "Potential state-changing/process operation", excerpt,
                 "Confirm it is a genuine side effect, advertised, and adequately bounded; subprocess alone is not a defect.","heuristic_screening")
        for m in list(OS_COMMAND_REGEX.finditer(text))[:2]:
            line = line_at(text,m.start())
            emit(skill, "R6.4", rel, line, "Possible OS-specific command", m.group(0),
                 "Check documented supported OS and alternate execution path; the command alone is not a portability defect.","heuristic_screening")
        for m in list(re.finditer(r"\b(?:verify\s*=\s*False|--no-verify|shell\s*=\s*True)\b",text))[:3]:
            line = line_at(text,m.start())
            emit(skill,"R5.2",rel,line,"Potential security-control bypass",m.group(0),
                 "shell=True is not necessarily a safety bypass; manually evaluate before labeling.","heuristic_screening")
        # Specific reproducibility candidate: year is discarded by a five-field cron format.
        if "def _to_cron" in text and re.search(r"f[\"'][^\n]*\{dt\.month\}[^\n]*\\?\*", text):
            line = next((i for i,l in enumerate(text.splitlines(),1) if "dt.month" in l), 0)
            emit(skill, "P5", rel, line, "One-off reminder cron expression omits year", text.splitlines()[line-1] if line else "",
                 "Candidate only: reproduce with a future-year scheduling test in a sandbox before confirming.","targeted_screening")
RUN_CHECKS.update(["P2", "P4", "R6.4", "P5"])
print("After full-folder screening:", len(CANDIDATES), "raw candidate signals")

## 7 — Deduplicate candidates and generate taxonomy coverage

Each candidate remains a separate rule/file/location instance. Repeated identical signals collapse by `skill_id, rule_id, file_path, line_number`. Rules not screened are marked **manual_only** or **not_evaluable**; they are **not** silently treated as passes.

In [ ]:
FIELDS = ["candidate_id", "skill_id", "repo", "skill_path", "stratum", "rule_id", "rule_name", "rule_type", "file_path", "line_number", "signal", "supporting_excerpt", "analysis_note", "detection_method", "verification_level", "candidate_status", "impact", "recommended_fix", "investigation_notes"]
seen = set(); deduped=[]
for x in CANDIDATES:
    key = (x["skill_id"],x["rule_id"],x["file_path"],x["line_number"],x["signal"])
    if key in seen: continue
    seen.add(key)
    x["candidate_id"] = hashlib.sha256("|".join(map(str,key)).encode()).hexdigest()[:12]
    deduped.append(x)
candidates = pd.DataFrame(deduped,columns=FIELDS).sort_values(["skill_id","rule_id","file_path","line_number"]).reset_index(drop=True)

manual_codes = {"R1.3","R1.4","R1.6","R2.2","R2.3","R4.1","R4.3","R5.3","R7.1","R7.2","R7.3","P3"}
screen_only = {"R3.3","R5.2","R5.6","R6.4","P2","P4","P5"}
coverage=[]
for rule in ALL_RULES:
    rid=rule["id"]
    if rid == "S2" and yaml is None:
        status="not_evaluable"
    elif rid == "P1" and not any(s["folder_complete"] and s["integrity_ok"] for s in SKILLS):
        status="not_evaluable"
    elif rid in RUN_CHECKS:
        status="screening_only" if rid in screen_only else "implemented_static"
    else:
        status="manual_only"
    coverage.append({"rule_id":rid,"rule_name":rule["name"],"source":rule.get("origin", ""),
                     "planned_mode":rule.get("mode", "static" if rid.startswith("S") else "manual"),
                     "coverage_status":status,"candidates":int((candidates.rule_id == rid).sum()),
                     "trigger":rule.get("trigger",""),"exclusion":rule.get("exclusion","")})
coverage_df=pd.DataFrame(coverage)
display(coverage_df[["rule_id","rule_name","coverage_status","candidates"]])
print("Deduplicated candidates:",len(candidates), "| skills with warnings:", candidates.skill_id.nunique(), "| manual-only rules:",(coverage_df.coverage_status=="manual_only").sum())

## 8 — Export results (preserves existing researcher assessment entries)

Outputs go to `results/rq2/` rather than Adam's `results/manual_review/` directory. Re-running carries forward the four researcher assessment fields using stable `candidate_id`. Do not store actual secrets or unsafe code payloads in these CSVs.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
candidate_path = OUT_DIR / "rq2_defect_candidates.csv"
HUMAN = ["candidate_status", "impact", "recommended_fix", "investigation_notes"]
if candidate_path.exists():
    prior = pd.read_csv(candidate_path,keep_default_na=False)
    if "candidate_id" in prior.columns:
        lookup = prior.drop_duplicates("candidate_id").set_index("candidate_id")
        for col in HUMAN:
            if col in lookup.columns:
                saved = candidates.candidate_id.map(lookup[col]).fillna("")
                candidates[col] = [old if str(old).strip() else new for old,new in zip(saved,candidates[col])]
candidates.to_csv(candidate_path,index=False)
coverage_df.to_csv(OUT_DIR / "rq2_taxonomy_coverage.csv",index=False)
INV.to_csv(OUT_DIR / "rq2_skill_inventory.csv",index=False)
counts = candidates.groupby(["rule_id","rule_name"]).agg(candidate_locations=("candidate_id","count"),skills_affected=("skill_id","nunique")).reset_index() if len(candidates) else pd.DataFrame(columns=["rule_id","rule_name","candidate_locations","skills_affected"])
counts.to_csv(OUT_DIR / "rq2_rule_summary.csv",index=False)

meta={"generated_at_utc":datetime.now(timezone.utc).isoformat(),"source_run":SOURCE_RUN,"taxonomy_version":TAXONOMY["version"],
      "taxonomy_sha256":hashlib.sha256(TAXONOMY_FILE.read_bytes()).hexdigest(),"exported_skill_count":len(SKILLS),
      "complete_export_count":int(INV.folder_complete.sum()),"integrity_verified_count":int(INV.integrity_ok.sum()),
      "candidate_locations":len(candidates),"skills_flagged":int(candidates.skill_id.nunique()),
      "not_confirmation":"Every warning is a candidate; human review required."}
(OUT_DIR / "rq2_run_metadata.json").write_text(json.dumps(meta,indent=2),encoding="utf-8")

# Intentionally avoid LaTeX raw excerpts: possible special characters and sensitive content.
macros = ["% Auto-generated from 06_rq2_defect_detection.ipynb; candidate warnings are not confirmed defects.",
          rf"\newcommand{{\rqTwoSkillsScreened}}{{{len(SKILLS)}}}",
          rf"\newcommand{{\rqTwoCompleteExports}}{{{int(INV.folder_complete.sum())}}}",
          rf"\newcommand{{\rqTwoCandidateLocations}}{{{len(candidates)}}}",
          rf"\newcommand{{\rqTwoSkillsFlagged}}{{{int(candidates.skill_id.nunique())}}}"]
(OUT_DIR / "rq2_auto_macros.tex").write_text("\n".join(macros)+"\n",encoding="utf-8")

report=["# RQ2 preliminary static-screening report", "",
        "**All findings are candidates, not confirmed defects.**", "",
        f"- Skills with exported manifests: **{len(SKILLS)}**",
        f"- Complete exports: **{int(INV.folder_complete.sum())}**",
        f"- Hash-verified exports: **{int(INV.integrity_ok.sum())}**",
        f"- Candidate rule/locations: **{len(candidates)}**",
        f"- Skills with at least one candidate: **{int(candidates.skill_id.nunique())}**",
        f"- Manual-only / non-evaluated checks: **{int(coverage_df.coverage_status.isin(['manual_only','not_evaluable']).sum())}**", "",
        "## Counts by rule", "", "| Rule | Candidate locations | Skills |", "|---|---:|---:|"]
for r in counts.itertuples(index=False):
    report.append(f"| `{r.rule_id}` | {r.candidate_locations} | {r.skills_affected} |")
report += ["", "## Caveats", "", "- Folders with incomplete or mismatched exports cannot support missing-file conclusions.",
           "- Heuristic alerts may be false positives, and manual-only rules have not been assessed.",
           "- Script content was inspected statically; nothing was executed.",
           "- Findings are not interchangeable with the Cisco RQ1 security findings.",
           "- Use the candidate CSV for independent manual validation and repairs."]
(OUT_DIR / "rq2_screening_report.md").write_text("\n".join(report)+"\n",encoding="utf-8")
print("Output folder:",OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file(): print(" -",p.name)
display(counts)

## 9 — What to do next (RQ2 researcher validation)

1. Investigate candidate findings and use `python scripts/rq2_edit.py --ids <candidate_id>` to save `candidate_status`, `impact`, `recommended_fix`, and `investigation_notes` in the existing CSV. These are the researcher investigation records.
2. For high-value candidates, create **safe reproducible tests** in an isolated environment. Never execute untrusted skill scripts directly on your laptop.
3. Review **`rq2_taxonomy_coverage.csv`** for categories not automated—especially instruction/implementation contradictions (`P3`), routing quality, persona/scope, and error handling.
4. For the paper, use `rq2_auto_macros.tex` via `\input{...}`. These macros describe **candidate counts only**, not validated defect prevalence.

**Example next experiment:** The `set-reminder` cron expression omits the year. Verify with a pure calculation and a controlled test before labeling it a confirmed functional defect. Keep this issue separate from Cisco's scanner warnings.